# AWS Pruning + SageMaker Inference Demo

This notebook demonstrates an end-to-end workflow:

1. Set up and validate AWS infrastructure access with the AWS CLI.
2. Run model pruning + artifact registration to S3.
3. Deploy (or update) a SageMaker endpoint.
4. Invoke prompts and read generation metadata.
5. Download token-level logits artifacts and compute token log probabilities.

The default calibration dataset source is `humanevalplus_prompts`, but the workflow is dataset-agnostic through a single `CALIBRATION_SOURCE` variable.

> Cost note: SageMaker endpoints and GPU instances are expensive. Use the cleanup section at the end when done.

In [ ]:
from __future__ import annotations

import io
import json
import os
import re
import shlex
import subprocess
import time
from pathlib import Path
from typing import Any

import boto3
import numpy as np
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd()


def run_cmd(command: str, env: dict[str, str] | None = None) -> str:
    """Run a shell command and return stdout.

    Parameters
    ----------
    command:
        Shell command to execute.
    env:
        Optional environment variables to merge with current process env.

    Returns
    -------
    str
        Captured standard output text.

    Raises
    ------
    RuntimeError
        If the command exits with a non-zero code.
    """

    print(f"$ {command}")
    merged_env = dict(os.environ)
    if env:
        merged_env.update(env)
    completed = subprocess.run(
        command,
        shell=True,
        check=False,
        cwd=REPO_ROOT,
        env=merged_env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}: {command}"
        )
    return completed.stdout


def parse_s3_uri(s3_uri: str) -> tuple[str, str]:
    """Split an S3 URI into (bucket, key)."""

    cleaned = s3_uri.replace("s3://", "", 1)
    if "/" not in cleaned:
        return cleaned, ""
    bucket, key = cleaned.split("/", 1)
    return bucket, key


def log_softmax(logits: np.ndarray) -> np.ndarray:
    """Compute row-wise numerically stable log-softmax."""

    max_per_row = np.max(logits, axis=1, keepdims=True)
    shifted = logits - max_per_row
    logsumexp = np.log(np.sum(np.exp(shifted), axis=1, keepdims=True))
    return shifted - logsumexp


print(f"Repo root: {REPO_ROOT}")

In [ ]:
# AWS and SageMaker parameters
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
AWS_ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "123456789012")
SAGEMAKER_ROLE_ARN = os.environ.get(
    "SAGEMAKER_ROLE_ARN",
    f"arn:aws:iam::{AWS_ACCOUNT_ID}:role/service-role/AmazonSageMaker-ExecutionRole",
)

PRUNING_ARTIFACT_BUCKET = os.environ.get("PRUNING_ARTIFACT_BUCKET", "my-pruning-artifacts")
PRUNING_ARTIFACT_PREFIX = os.environ.get("PRUNING_ARTIFACT_PREFIX", "qwen-pruning")
PRUNING_LOGITS_BUCKET = os.environ.get("PRUNING_LOGITS_BUCKET", PRUNING_ARTIFACT_BUCKET)
PRUNING_LOGITS_PREFIX = os.environ.get("PRUNING_LOGITS_PREFIX", "logits")

PRUNING_ENDPOINT_NAME = os.environ.get("PRUNING_ENDPOINT_NAME", "qwen-pruning-endpoint")
PRUNING_INSTANCE_TYPE = os.environ.get("PRUNING_INSTANCE_TYPE", "ml.g5.48xlarge")
PRUNING_INSTANCE_COUNT = int(os.environ.get("PRUNING_INSTANCE_COUNT", "1"))

BASE_MODEL_ID = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen2.5-Coder-7B-Instruct")
PRUNING_LEVELS = os.environ.get("PRUNING_LEVELS", "0,20")

# Switch this one variable to change calibration data.
# Options include:
# - humanevalplus_prompts
# - hf:<dataset_name>:<split>:<text_field>
CALIBRATION_SOURCE = os.environ.get(
    "CALIBRATION_SOURCE", "humanevalplus_prompts"
)

MAX_CALIBRATION_SAMPLES = int(os.environ.get("MAX_CALIBRATION_SAMPLES", "16"))
MAX_CALIBRATION_TOKENS = int(os.environ.get("MAX_CALIBRATION_TOKENS", "512"))

# Serving image settings
ECR_REPOSITORY_NAME = os.environ.get("ECR_REPOSITORY_NAME", "qwen-serving")
IMAGE_TAG = os.environ.get("IMAGE_TAG", "latest")
CONTAINER_IMAGE_URI = (
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
    f"{ECR_REPOSITORY_NAME}:{IMAGE_TAG}"
)

PARAMETERS = {
    "AWS_REGION": AWS_REGION,
    "AWS_ACCOUNT_ID": AWS_ACCOUNT_ID,
    "SAGEMAKER_ROLE_ARN": SAGEMAKER_ROLE_ARN,
    "PRUNING_ARTIFACT_BUCKET": PRUNING_ARTIFACT_BUCKET,
    "PRUNING_ARTIFACT_PREFIX": PRUNING_ARTIFACT_PREFIX,
    "PRUNING_LOGITS_BUCKET": PRUNING_LOGITS_BUCKET,
    "PRUNING_LOGITS_PREFIX": PRUNING_LOGITS_PREFIX,
    "PRUNING_ENDPOINT_NAME": PRUNING_ENDPOINT_NAME,
    "PRUNING_INSTANCE_TYPE": PRUNING_INSTANCE_TYPE,
    "PRUNING_INSTANCE_COUNT": str(PRUNING_INSTANCE_COUNT),
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "PRUNING_LEVELS": PRUNING_LEVELS,
    "CALIBRATION_SOURCE": CALIBRATION_SOURCE,
}

print(json.dumps(PARAMETERS, indent=2))

## 1) Validate AWS CLI and Identity

These commands verify that your local AWS CLI is configured and that credentials can access your target account/region.

In [ ]:
run_cmd("aws --version")
identity_output = run_cmd("aws sts get-caller-identity --output json")
identity = json.loads(identity_output)
print("Caller identity:")
print(json.dumps(identity, indent=2))

configured_region = run_cmd("aws configure get region").strip()
print(f"Configured AWS CLI region: {configured_region!r}")
if configured_region and configured_region != AWS_REGION:
    print(
        "Warning: notebook AWS_REGION differs from AWS CLI configured region. "
        "Commands below pass --region explicitly."
    )

In [ ]:
# Optional: create S3 buckets if they do not exist in your account.
# You can skip this cell if buckets already exist.
for bucket_name in sorted({PRUNING_ARTIFACT_BUCKET, PRUNING_LOGITS_BUCKET}):
    check_cmd = f"aws s3api head-bucket --bucket {shlex.quote(bucket_name)}"
    try:
        run_cmd(check_cmd)
        print(f"Bucket exists and is accessible: {bucket_name}")
    except RuntimeError:
        print(f"Creating bucket: {bucket_name}")
        if AWS_REGION == "us-east-1":
            create_cmd = f"aws s3api create-bucket --bucket {shlex.quote(bucket_name)}"
        else:
            create_cmd = (
                "aws s3api create-bucket "
                f"--bucket {shlex.quote(bucket_name)} "
                f"--region {shlex.quote(AWS_REGION)} "
                "--create-bucket-configuration "
                f"LocationConstraint={shlex.quote(AWS_REGION)}"
            )
        run_cmd(create_cmd)

## 2) Configure Calibration Dataset Source

`CALIBRATION_SOURCE` controls pruning calibration data. Keep this notebook generic by switching only that single variable.

Examples:
- `humanevalplus_prompts` (default)
- `hf:openai_humaneval:test:prompt`
- `hf:mbpp:test:text`

In [ ]:
print(f"Using calibration source: {CALIBRATION_SOURCE}")
if not (
    CALIBRATION_SOURCE == "humanevalplus_prompts"
    or CALIBRATION_SOURCE.startswith("hf:")
):
    raise ValueError(
        "CALIBRATION_SOURCE must be 'humanevalplus_prompts' or start with 'hf:'."
    )

## 3) Prune and Register Model Artifacts

This step can be expensive and slow for large models. For initial smoke tests, keep:
- `MAX_CALIBRATION_SAMPLES` low (for example 4-16)
- `MAX_CALIBRATION_TOKENS` low (for example 256-512)
- fewer pruning levels (for example `0,20`)

In [ ]:
runtime_env = {
    "AWS_REGION": AWS_REGION,
    "AWS_DEFAULT_REGION": AWS_REGION,
    "SAGEMAKER_ROLE_ARN": SAGEMAKER_ROLE_ARN,
    "PRUNING_ARTIFACT_BUCKET": PRUNING_ARTIFACT_BUCKET,
    "PRUNING_ARTIFACT_PREFIX": PRUNING_ARTIFACT_PREFIX,
    "PRUNING_LOGITS_BUCKET": PRUNING_LOGITS_BUCKET,
    "PRUNING_LOGITS_PREFIX": PRUNING_LOGITS_PREFIX,
    "PRUNING_ENDPOINT_NAME": PRUNING_ENDPOINT_NAME,
    "PRUNING_INSTANCE_TYPE": PRUNING_INSTANCE_TYPE,
    "PRUNING_INSTANCE_COUNT": str(PRUNING_INSTANCE_COUNT),
}

prune_command = " ".join(
    [
        "python",
        "infra/aws/sagemaker/prune_and_register.py",
        f"--base-model-id {shlex.quote(BASE_MODEL_ID)}",
        f"--calibration-source {shlex.quote(CALIBRATION_SOURCE)}",
        f"--pruning-levels {shlex.quote(PRUNING_LEVELS)}",
        f"--max-calibration-samples {MAX_CALIBRATION_SAMPLES}",
        f"--max-calibration-tokens {MAX_CALIBRATION_TOKENS}",
        f"--artifact-bucket {shlex.quote(PRUNING_ARTIFACT_BUCKET)}",
        f"--artifact-prefix {shlex.quote(PRUNING_ARTIFACT_PREFIX)}",
        f"--region {shlex.quote(AWS_REGION)}",
    ]
)

prune_stdout = run_cmd(prune_command, env=runtime_env)
manifest_match = re.search(r"\{[\s\S]*\}", prune_stdout)
if not manifest_match:
    raise RuntimeError("Could not parse manifest JSON from prune script output.")
manifest_payload = json.loads(manifest_match.group(0))
MANIFEST_S3_URI = manifest_payload["manifest_s3_uri"]
print(f"Manifest uploaded to: {MANIFEST_S3_URI}")

## 4) Build and Push Serving Image (AWS CLI)

If your serving image is not already in ECR, run these commands.

If you already have a suitable `CONTAINER_IMAGE_URI`, you can skip this step.

In [ ]:
ecr_login_cmd = (
    f"aws ecr get-login-password --region {shlex.quote(AWS_REGION)} | "
    f"docker login --username AWS --password-stdin "
    f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com"
)
create_repo_cmd = (
    "aws ecr describe-repositories "
    f"--repository-names {shlex.quote(ECR_REPOSITORY_NAME)} "
    f"--region {shlex.quote(AWS_REGION)}"
)
create_repo_fallback_cmd = (
    "aws ecr create-repository "
    f"--repository-name {shlex.quote(ECR_REPOSITORY_NAME)} "
    f"--region {shlex.quote(AWS_REGION)}"
)
build_cmd = (
    "docker build "
    "-t qwen-serving-local "
    "-f infra/containers/qwen-serving/Dockerfile "
    "infra/containers/qwen-serving"
)
tag_cmd = f"docker tag qwen-serving-local {shlex.quote(CONTAINER_IMAGE_URI)}"
push_cmd = f"docker push {shlex.quote(CONTAINER_IMAGE_URI)}"

print("Run these commands if needed:")
print(ecr_login_cmd)
print(create_repo_cmd)
print("If describe-repositories fails, run:")
print(create_repo_fallback_cmd)
print(build_cmd)
print(tag_cmd)
print(push_cmd)
print(f"Final image URI: {CONTAINER_IMAGE_URI}")

## 5) Deploy or Update SageMaker Endpoint

This calls the project script that creates a model + endpoint config and then creates/updates the endpoint name.

In [ ]:
if "MANIFEST_S3_URI" not in globals() or not MANIFEST_S3_URI:
    raise RuntimeError("Run pruning step first so MANIFEST_S3_URI is set.")

deploy_command = " ".join(
    [
        "python",
        "infra/aws/sagemaker/deploy_endpoint.py",
        f"--container-image-uri {shlex.quote(CONTAINER_IMAGE_URI)}",
        f"--manifest-s3-uri {shlex.quote(MANIFEST_S3_URI)}",
        f"--endpoint-name {shlex.quote(PRUNING_ENDPOINT_NAME)}",
        f"--role-arn {shlex.quote(SAGEMAKER_ROLE_ARN)}",
        f"--instance-type {shlex.quote(PRUNING_INSTANCE_TYPE)}",
        f"--instance-count {PRUNING_INSTANCE_COUNT}",
        f"--region {shlex.quote(AWS_REGION)}",
        f"--logits-bucket {shlex.quote(PRUNING_LOGITS_BUCKET)}",
        f"--logits-prefix {shlex.quote(PRUNING_LOGITS_PREFIX)}",
    ]
)
run_cmd(deploy_command, env=runtime_env)

In [ ]:
def wait_for_endpoint_in_service(
    endpoint_name: str,
    region: str,
    timeout_seconds: int = 60 * 30,
    poll_seconds: int = 30,
) -> str:
    """Wait until endpoint enters InService or a terminal failure state."""

    client = boto3.client("sagemaker", region_name=region)
    started = time.time()
    while True:
        response = client.describe_endpoint(EndpointName=endpoint_name)
        status = response["EndpointStatus"]
        print(f"Endpoint status: {status}")
        if status == "InService":
            return status
        if status in {"Failed", "OutOfService"}:
            raise RuntimeError(
                f"Endpoint entered terminal state: {status}\n"
                f"Failure reason: {response.get('FailureReason', 'unknown')}"
            )
        if time.time() - started > timeout_seconds:
            raise TimeoutError("Timed out waiting for endpoint to become InService.")
        time.sleep(poll_seconds)


wait_for_endpoint_in_service(PRUNING_ENDPOINT_NAME, AWS_REGION)

## 6) Invoke the Endpoint with a Prompt

This section demonstrates a direct `boto3` invoke so you can keep everything in one notebook execution context.

In [ ]:
runtime_client = boto3.client("sagemaker-runtime", region_name=AWS_REGION)

SAMPLE_PROMPT = (
    "Write a Python function `is_palindrome(s: str) -> bool` "
    "that ignores casing and non-alphanumeric characters."
)
REQUEST_TASK_ID = "demo-notebook-task-001"
REQUEST_PRUNING_LEVEL = int(PRUNING_LEVELS.split(",")[0])
REQUEST_SEED = 123

request_payload = {
    "prompt": SAMPLE_PROMPT,
    "task_id": REQUEST_TASK_ID,
    "pruning_level": REQUEST_PRUNING_LEVEL,
    "seed": REQUEST_SEED,
    "max_new_tokens": 128,
    "temperature": 0.0,
    "top_p": 1.0,
}

response = runtime_client.invoke_endpoint(
    EndpointName=PRUNING_ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(request_payload).encode("utf-8"),
)
response_payload = json.loads(response["Body"].read().decode("utf-8"))

print("Response metadata:")
print(
    json.dumps(
        {
            "task_id": response_payload.get("task_id"),
            "pruning_level": response_payload.get("pruning_level"),
            "token_count": response_payload.get("token_count"),
            "request_id": response_payload.get("request_id"),
            "logits_s3_uri": response_payload.get("logits_s3_uri"),
        },
        indent=2,
    )
)
print("\nGenerated text:\n")
print(response_payload.get("generated_text", ""))

## 7) Compute Token Log Probabilities from Logits Artifact

The serving container currently stores per-token logits at `logits_s3_uri` as JSONL rows. This cell also supports `.npz` artifacts for portability.

For each generated token, we compute:
- full-vocabulary log-softmax
- selected token log probability for the emitted token id

In [ ]:
def load_logits_artifact(s3_uri: str, region: str) -> tuple[np.ndarray, np.ndarray]:
    """Download and parse logits artifact from S3.

    Parameters
    ----------
    s3_uri:
        S3 URI from inference response metadata.
    region:
        AWS region used for S3 client.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Tuple of (generated_token_ids, logits_2d_array).
    """

    bucket, key = parse_s3_uri(s3_uri)
    s3 = boto3.client("s3", region_name=region)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    if key.endswith(".npz"):
        with np.load(io.BytesIO(body)) as npz_data:
            token_ids = np.array(npz_data["generated_token_ids"], dtype=np.int64)
            logits = np.array(npz_data["logits"], dtype=np.float32)
        return token_ids, logits

    lines = body.decode("utf-8").splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    token_ids = np.array([int(row["token_id"]) for row in rows], dtype=np.int64)
    logits = np.array([row["logits"] for row in rows], dtype=np.float32)
    return token_ids, logits


logits_s3_uri = response_payload.get("logits_s3_uri")
if not logits_s3_uri:
    raise RuntimeError("No logits_s3_uri in endpoint response.")

generated_token_ids, step_logits = load_logits_artifact(logits_s3_uri, AWS_REGION)
if generated_token_ids.size == 0 or step_logits.size == 0:
    raise RuntimeError("Logits artifact is empty; cannot compute log probabilities.")

step_logprobs = log_softmax(step_logits)
selected_logprobs = step_logprobs[
    np.arange(generated_token_ids.shape[0]), generated_token_ids
]

print(f"Loaded {generated_token_ids.shape[0]} generated tokens.")
print(
    "Average selected-token log probability: "
    f"{float(np.mean(selected_logprobs)):.4f}"
)
print(
    "Average selected-token probability: "
    f"{float(np.mean(np.exp(selected_logprobs))):.4f}"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
preview_count = min(25, generated_token_ids.shape[0])

print("First generated tokens with selected-token log probabilities:")
for index in range(preview_count):
    token_id = int(generated_token_ids[index])
    token_text = tokenizer.decode([token_id])
    token_logprob = float(selected_logprobs[index])
    token_prob = float(np.exp(token_logprob))
    print(
        f"idx={index:03d} token_id={token_id:>8} "
        f"logp={token_logprob:>8.4f} p={token_prob:>8.6f} "
        f"token={token_text!r}"
    )

In [ ]:
# Optional: invoke via project helper script instead of direct boto3.
# This can be useful for parity with command-line workflows.
invoke_script_cmd = " ".join(
    [
        "python",
        "infra/aws/sagemaker/invoke_endpoint.py",
        f"--endpoint-name {shlex.quote(PRUNING_ENDPOINT_NAME)}",
        f"--prompt {shlex.quote(SAMPLE_PROMPT)}",
        f"--task-id {shlex.quote(REQUEST_TASK_ID)}",
        f"--pruning-level {REQUEST_PRUNING_LEVEL}",
        f"--seed {REQUEST_SEED}",
        "--max-new-tokens 128",
        "--temperature 0.0",
        "--top-p 1.0",
        f"--region {shlex.quote(AWS_REGION)}",
    ]
)
print(invoke_script_cmd)
# run_cmd(invoke_script_cmd, env=runtime_env)

## 8) Cleanup and Cost Controls

Delete endpoint resources when you are done. Endpoint instances continue billing while active.

Recommended order:
1. Delete endpoint.
2. Delete endpoint config(s).
3. Delete model(s).
4. Optionally delete pruning/logits artifacts from S3.

In [ ]:
sm_client = boto3.client("sagemaker", region_name=AWS_REGION)

# Set RUN_CLEANUP=True only when you are certain you want to remove resources.
RUN_CLEANUP = False

if RUN_CLEANUP:
    print(f"Deleting endpoint: {PRUNING_ENDPOINT_NAME}")
    sm_client.delete_endpoint(EndpointName=PRUNING_ENDPOINT_NAME)

    # Optional: clean up stale endpoint configs/models created by this workflow.
    # We use name prefixes matching deploy_endpoint.py defaults.
    endpoint_configs = sm_client.list_endpoint_configs(
        NameContains="qwen-pruning-config"
    ).get("EndpointConfigs", [])
    for config in endpoint_configs:
        config_name = config["EndpointConfigName"]
        print(f"Deleting endpoint config: {config_name}")
        sm_client.delete_endpoint_config(EndpointConfigName=config_name)

    models = sm_client.list_models(NameContains="qwen-pruning-model").get("Models", [])
    for model in models:
        model_name = model["ModelName"]
        print(f"Deleting model: {model_name}")
        sm_client.delete_model(ModelName=model_name)

    print("Cleanup requests submitted.")
else:
    print(
        "Cleanup is disabled. Set RUN_CLEANUP=True to delete endpoint/config/model resources."
    )

In [ ]:
# Optional artifact cleanup command templates.
print("Artifact cleanup command templates:")
print(
    "aws s3 rm "
    f"s3://{PRUNING_ARTIFACT_BUCKET}/{PRUNING_ARTIFACT_PREFIX} "
    "--recursive"
)
print(
    "aws s3 rm "
    f"s3://{PRUNING_LOGITS_BUCKET}/{PRUNING_LOGITS_PREFIX} "
    "--recursive"
)